# Professional Healthcare Dataset Preprocessing Pipeline
This notebook cleans, filters, and consolidates four major medical datasets into a single unified JSON format.

In [1]:
!pip install -q datasets pandas regex tqdm

import re
import os
import json
import pandas as pd
from datasets import load_dataset, Dataset
from tqdm import tqdm

# Setup workspace
OUTPUT_DIR = '/content/medical_datasets'
os.makedirs(OUTPUT_DIR, exist_ok=True)

combined_medical_data = []
dataset_counts = {}

## 1. Utility Functions & Filters

In [2]:
def clean_text(text: str) -> str:
    if not isinstance(text, str): return ""
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def save_json(data, filename):
    filepath = os.path.join(OUTPUT_DIR, filename)
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f'Saved {len(data):,} records to {filepath}')

ELDERLY_KEYWORDS = [r'\belderly\b', r'\bgeriatric\b', r'\bolder adult', r'\baging\b', r'\bsenior\b']
keyword_regex = re.compile('|'.join(ELDERLY_KEYWORDS), flags=re.IGNORECASE)

def contains_elderly_context(text: str) -> bool:
    return bool(keyword_regex.search(text))

## 2. Dataset Processing

In [ ]:
print('Processing PubMedQA...')
try:
    pubmed_ds = load_dataset('fedml/PubMedQA_instruction', split='train')
    pubmed_list = [{'instruction': clean_text(r.get('instruction','')), 'input': clean_text(r.get('input','')), 'response': clean_text(r.get('output','')), 'category': 'Medical Q&A', 'source': 'PubMedQA'} for r in pubmed_ds]
    save_json(pubmed_list, 'PubMedQA.json')
    combined_medical_data.extend(pubmed_list)
    dataset_counts['PubMedQA'] = len(pubmed_list)
except Exception as e: print(f'PubMedQA Error: {e}')

Processing PubMedQA...


README.md:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

data/train-00000-of-00001-d9d142e5f3625d(…): reconstructing file:   0%|          |  0.00B /  274MB            

data/train-00000-of-00001-d9d142e5f3625d(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-83c33859e732305(…): reconstructing file:   0%|          |  0.00B /  986kB            

data/test-00000-of-00001-83c33859e732305(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/272518 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
print('Processing MedQuAD...')
try:
    medquad_ds = load_dataset('keivalya/MedQuad-MedicalQnADataset', split='train')
    medquad_list = [{'instruction': 'Answer the medical question.', 'input': clean_text(r.get('Question','')), 'response': clean_text(r.get('Answer','')), 'category': 'Medical Q&A', 'source': 'MedQuAD'} for r in medquad_ds]
    save_json(medquad_list, 'MedQuAD.json')
    combined_medical_data.extend(medquad_list)
    dataset_counts['MedQuAD'] = len(medquad_list)
except Exception as e: print(f'MedQuAD Error: {e}')

In [ ]:
print('Processing MedDialog...')
try:
    meddialog_ds = load_dataset('khoaliamle/MedDialog-EN-100k', split='train')
    meddialog_list = [{'instruction': 'Respond to medical concern.', 'input': clean_text(r.get('input','')), 'response': clean_text(r.get('output','')), 'category': 'Medical Dialogue', 'source': 'MedDialog'} for r in meddialog_ds]
    save_json(meddialog_list, 'MedDialog.json')
    combined_medical_data.extend(meddialog_list)
    dataset_counts['MedDialog'] = len(meddialog_list)
except Exception as e: print(f'MedDialog Error: {e}')

In [ ]:
print('Processing HealthCareMagic...')
try:
    hcm_ds = load_dataset('lavita/ChatDoctor-HealthCareMagic-100k', split='train')
    hcm_list = [{'instruction': 'Provide consultation.', 'input': clean_text(r.get('input','')), 'response': clean_text(r.get('output','')), 'category': 'Medical Dialogue', 'source': 'HealthCareMagic'} for r in hcm_ds]
    save_json(hcm_list, 'HealthCareMagic.json')
    combined_medical_data.extend(hcm_list)
    dataset_counts['HealthCareMagic'] = len(hcm_list)
except Exception as e: print(f'HealthCareMagic Error: {e}')

## 3. Consolidation & Final Export

In [ ]:
print('\nMerging all datasets...')
save_json(combined_medical_data, 'medical.json')

print('\n' + '='*45)
print('FINAL PROCESSING SUMMARY')
print('='*45)
for name, count in dataset_counts.items():
    print(f'{name:<20}: {count:>10,} samples')
print('-'*45)
print(f'Total Consolidated : {len(combined_medical_data):>10,} samples')
print('='*45)

In [ ]:
!pip install -q datasets pandas regex tqdm

import re
import pandas as pd
from datasets import load_dataset, Dataset
from tqdm import tqdm

# Define elderly/geriatric keywords for targeted domain filtering
ELDERLY_KEYWORDS = [
    r'\belderly\b', r'\bgeriatric\b', r'\bolder adult', r'\baging\b',
    r'\bsenior\b', r'\balzheimer', r'\bdementia\b', r'\bparkinson', r'\bosteoporosis\b',
    r'\bfrail\b', r'\bfrailty\b', r'\bpresby\w*', r'\bage-related\b', r'\bmedicare\b',
    r'\bfalls?\b', r'\bcognitive decline\b', r'\bcaregiver\b', r'\bgerontology\b'
]

keyword_regex = re.compile('|'.join(ELDERLY_KEYWORDS), flags=re.IGNORECASE)

def contains_elderly_context(text: str) -> bool:
    """Check if the text contains any elderly/geriatric keywords."""
    if not isinstance(text, str):
        return False
    return bool(keyword_regex.search(text))

def clean_text(text: str) -> str:
    """Clean text by removing HTML tags, extra whitespace, and junk characters."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<[^>]+>', ' ', text)  # Clean HTML tags
    text = re.sub(r'\s+', ' ', text)      # Normalize spaces
    text = text.strip()
    return text

processed_datasets = []

### Processing PubMedQA (fedml/PubMedQA_instruction)

In [ ]:
print("--> Processing PubMedQA...")
try:
    ds_pubmed = load_dataset("fedml/PubMedQA_instruction", split="train")
    pubmed_list = []

    for row in tqdm(ds_pubmed):
        # Extract fields
        instruction = clean_text(row.get("instruction", "Answer the following medical question using the provided context."))
        input_text = clean_text(row.get("input", ""))
        output_text = clean_text(row.get("output", ""))

        full_text = f"{instruction} {input_text} {output_text}"

        # Filter for elderly care domain
        if contains_elderly_context(full_text):
            pubmed_list.append({
                "source": "PubMedQA",
                "instruction": instruction if instruction else "Answer the medical research query based on context.",
                "input": input_text,
                "response": output_text, # Changed to 'response' as per initial request
                "category": "Medical Q&A", # Added 'category'
            })

    df_pubmed = pd.DataFrame(pubmed_list)
    print(f"PubMedQA Elderly Samples: {len(df_pubmed)}")
    processed_datasets.append(df_pubmed)
except Exception as e:
    print(f"Failed to load/process PubMedQA: {e}")

--> Processing PubMedQA...


README.md:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

data/train-00000-of-00001-d9d142e5f3625d(…): reconstructing file:   0%|          |  0.00B /  274MB            

data/train-00000-of-00001-d9d142e5f3625d(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-83c33859e732305(…): reconstructing file:   0%|          |  0.00B /  986kB            

data/test-00000-of-00001-83c33859e732305(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/272518 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

100%|██████████| 272518/272518 [00:26<00:00, 10423.05it/s]

PubMedQA Elderly Samples: 7251


### Processing MedQuAD (keivalya/MedQuad-MedicalQnADataset)

In [ ]:
print("\n--> Processing MedQuAD...")
try:
    ds_medquad = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")
    medquad_list = []

    for row in tqdm(ds_medquad):
        question = clean_text(row.get("Question", ""))
        answer = clean_text(row.get("Answer", ""))
        focus = clean_text(row.get("Focus", ""))

        full_text = f"{focus} {question} {answer}"

        if contains_elderly_context(full_text):
            medquad_list.append({
                "source": "MedQuAD",
                "instruction": f"Provide detailed medical information regarding {focus}." if focus else "Answer the medical query.",
                "input": question,
                "response": answer, # Changed to 'response'
                "category": "Medical Q&A", # Added 'category'
            })

    df_medquad = pd.DataFrame(medquad_list)
    print(f"MedQuAD Elderly Samples: {len(df_medquad)}")
    processed_datasets.append(df_medquad)
except Exception as e:
    print(f"Failed to load/process MedQuAD: {e}")


--> Processing MedQuAD...


README.md:   0%|          | 0.00/233 [00:00<?, ?B/s]

medDataset_processed.csv: reconstructing file:   0%|          |  0.00B / 22.5MB            

medDataset_processed.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16407 [00:00<?, ? examples/s]

100%|██████████| 16407/16407 [00:06<00:00, 2554.25it/s]

MedQuAD Elderly Samples: 1117


### Processing HealthCareMagic QA (lavita/ChatDoctor-HealthCareMagic-100k)

In [ ]:
print("\n--> Processing HealthCareMagic QA...")
try:
    ds_hcm = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")
    hcm_list = []

    for row in tqdm(ds_hcm):
        patient_input = clean_text(row.get("input", ""))
        doctor_output = clean_text(row.get("output", ""))

        full_text = f"{patient_input} {doctor_output}"

        if contains_elderly_context(full_text):
            hcm_list.append({
                "source": "HealthCareMagic",
                "instruction": "Provide a medical consultation and response based on the patient's described symptoms.",
                "input": patient_input,
                "response": doctor_output, # Changed to 'response'
                "category": "Medical Dialogue", # Added 'category'
            })

    df_hcm = pd.DataFrame(hcm_list)
    print(f"HealthCareMagic Elderly Samples: {len(df_hcm)}")
    processed_datasets.append(df_hcm)
except Exception as e:
    print(f"Failed to load/process HealthCareMagic: {e}")


--> Processing HealthCareMagic QA...


README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…): reconstructing file:   0%|          |  0.00B / 70.5MB            

data/train-00000-of-00001-5e7cb295b9cff0(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

100%|██████████| 112165/112165 [00:35<00:00, 3160.89it/s]

HealthCareMagic Elderly Samples: 4979


### Processing MedDialog (UCSD26/medical_dialog)

In [ ]:
print("\n--> Processing MedDialog...")
try:
    ds_dialog = load_dataset("UCSD26/medical_dialog", split="train")
    dialog_list = []

    for row in tqdm(ds_dialog):
        # MedDialog fields often vary depending on HF format mirror (description/utterance/dialogue)
        context = clean_text(row.get("description", row.get("utterance", "")))
        dialogue = clean_text(row.get("dialogue", row.get("formal_description", "")))

        full_text = f"{context} {dialogue}"

        if contains_elderly_context(full_text):
            dialog_list.append({
                "source": "MedDialog",
                "instruction": "Analyze the patient medical dialogue and provide clinical recommendations.",
                "input": context if context else dialogue[:200],
                "response": dialogue if context else dialogue[200:], # Changed to 'response'
                "category": "Medical Dialogue", # Added 'category'
            })

    df_dialog = pd.DataFrame(dialog_list)
    print(f"MedDialog Elderly Samples: {len(df_dialog)}")
    processed_datasets.append(df_dialog)
except Exception as e:
    print(f"Skipping or handled alternative MedDialog format: {e}")


--> Processing MedDialog...


README.md:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

medical_dialog.py:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Skipping or handled alternative MedDialog format: Dataset scripts are no longer supported, but found medical_dialog.py


### Unifying, Final Cleaning, and Exporting Datasets

In [ ]:
import pandas as pd

print("\n--> Combining Datasets...")
final_df = pd.concat(processed_datasets, ignore_index=True)

# Drop missing values and duplicates across input/output
final_df.dropna(subset=["input", "response"], inplace=True) # Changed 'output' to 'response'
final_df.drop_duplicates(subset=["input", "response"], inplace=True) # Changed 'output' to 'response'

# Remove trivial/extremely short inputs or outputs
final_df = final_df[(final_df['input'].str.len() > 10) & (final_df['response'].str.len() > 10)] # Changed 'output' to 'response'

print(f"\nFinal Consolidated Elderly Dataset Size: {len(final_df)} records")
print("\nSample Distribution by Source:")
print(final_df['source'].value_counts())

# Export to JSON and Hugging Face Dataset format
# Renaming for clarity as per the initial request of 'medical.json'
final_df.to_json("medical.json", orient="records", indent=2)
hf_dataset = Dataset.from_pandas(final_df)
hf_dataset.save_to_disk("./elderly_dataset_hf")

print("\nSaved output files:")
print("1. medical.json")
print("2. ./elderly_dataset_hf (Hugging Face Dataset folder)")


--> Combining Datasets...

Final Consolidated Elderly Dataset Size: 6083 records

Sample Distribution by Source:
source
HealthCareMagic    4971
MedQuAD            1112
Name: count, dtype: int64


Saving the dataset (0/1 shards):   0%|          | 0/6083 [00:00<?, ? examples/s]


Saved output files:
1. medical.json
2. ./elderly_dataset_hf (Hugging Face Dataset folder)


### Adding Agent-Generated Medical Condition Entries

Based on your request, I will now parse the provided markdown text to extract individual medical conditions and their descriptions. These will be structured and added as new entries to the `final_df` DataFrame, enriching the dataset with specific information on geriatric health conditions. Finally, the updated dataset will be re-exported to `medical.json` and the Hugging Face dataset format.

In [ ]:
markdown_text_to_add = """
As the body ages, natural cellular wear, immune system changes, and vascular stiffening make older adults more susceptible to specific health conditions. These conditions generally fall into a few key categories:

---

## 1. Neurodegenerative & Cognitive Disorders

* **Alzheimer's Disease & Dementia:** Progressive cognitive decline affecting memory, reasoning, and daily functioning. Age is the strongest known risk factor.
* **Parkinson's Disease:** A nervous system disorder that leads to tremors, muscle rigidity, slowed movement, and balance issues.

## 2. Musculoskeletal Conditions

* **Osteoarthritis:** Breakdown of joint cartilage resulting in chronic pain, stiffness, and reduced mobility.
* **Osteoporosis:** Loss of bone density, making bones brittle and significantly increasing the risk of serious fractures from minor falls.

## 3. Cardiovascular & Metabolic Diseases

* **Hypertension (High Blood Pressure):** Extremely common due to the natural stiffening of blood vessels.
* **Heart Disease:** Includes coronary artery disease, heart failure, and arrhythmias.
* **Type 2 Diabetes:** Rates increase with age as insulin sensitivity decreases and pancreatic function changes.
* **Stroke:** Brain tissue damage caused by blocked or ruptured blood vessels.

## 4. Sensory & Respiratory Impairments

* **Sensory Loss:** Cataracts, glaucoma, age-related macular degeneration (AMD), and presbycusis (age-related hearing loss).
* **COPD (Chronic Obstructive Pulmonary Disease):** Long-term, progressive lung damage causing breathlessness and coughing.

## 5. Mental Health & General Wellbeing

* **Depression & Anxiety:** Often triggered or exacerbated by chronic pain, isolation, loss of independence, or major life shifts.

---

> **Note:** Many older adults manage multiple chronic conditions simultaneously (multimorbidity). Regular health screenings, light aerobic exercise, strength training, and a nutrient-dense diet are crucial for maintaining functional independence.
"""

def parse_medical_markdown(markdown_content: str) -> list:
    """
    Parses the provided markdown content to extract medical conditions and their descriptions,
    structured for a DataFrame.
    """
    entries = []
    # Use re.split to separate categories, keeping the category header for later extraction
    category_blocks = re.split(r'(## \d+\. .*?)\n', markdown_content)

    # Filter out empty strings and the initial descriptive text before the first category
    category_blocks = [block.strip() for block in category_blocks if block.strip()]

    current_category = None
    for block in category_blocks:
        # Check if the block is a category header
        category_match = re.match(r'## \d+\. (.*)', block)
        if category_match:
            current_category = category_match.group(1).strip()
        elif current_category:
            # If it's not a category header but we have a current category, it must be content for that category
            # Find all condition bullet points within this block
            condition_matches = re.finditer(r'\* \*\*(.*?):\*\* (.*)', block)
            for match in condition_matches:
                condition_name = match.group(1).strip()
                description = match.group(2).strip()

                entries.append({
                    "source": "Agent Generated Medical Conditions",
                    "instruction": f"Provide information about {condition_name}.",
                    "input": condition_name,
                    "response": description,
                    "category": current_category,
                })
    return entries

# Parse the markdown text
new_entries_list = parse_medical_markdown(markdown_text_to_add)

# Create a DataFrame from the new entries
df_new_entries = pd.DataFrame(new_entries_list)

# Add to the processed_datasets list
print(f"\nAdding {len(df_new_entries)} new entries from markdown to processed_datasets.")
processed_datasets.append(df_new_entries)

# The final consolidation and export will happen in the next cell (d8a2bc26)
# No need to re-export here, as the final_df will be re-created.


Adding 11 new entries from markdown to processed_datasets.


### Processing HealthCareMagic (RafaelMPereira/HealthCareMagic-100k-Chat-Format-en)

In [ ]:
print("\n--> Processing RafaelMPereira/HealthCareMagic-100k-Chat-Format-en...")
try:
    ds_hcm_chat = load_dataset("RafaelMPereira/HealthCareMagic-100k-Chat-Format-en", split="train")
    hcm_chat_list = []

    for row in tqdm(ds_hcm_chat):
        patient_input = clean_text(row.get("input", ""))
        doctor_output = clean_text(row.get("output", ""))

        full_text = f"{patient_input} {doctor_output}"

        if contains_elderly_context(full_text):
            hcm_chat_list.append({
                "source": "HealthCareMagic (Chat Format)",
                "instruction": "Provide a medical consultation and response based on the patient's described symptoms.",
                "input": patient_input,
                "response": doctor_output,
                "category": "Medical Dialogue",
            })

    df_hcm_chat = pd.DataFrame(hcm_chat_list)
    print(f"HealthCareMagic (Chat Format) Elderly Samples: {len(df_hcm_chat)}")
    processed_datasets.append(df_hcm_chat)
except Exception as e:
    print(f"Failed to load/process RafaelMPereira/HealthCareMagic-100k-Chat-Format-en: {e}")


--> Processing RafaelMPereira/HealthCareMagic-100k-Chat-Format-en...


README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

HealthCareMagic-100k-en.jsonl: reconstructing file:   0%|          |  0.00B /  125MB            

HealthCareMagic-100k-en.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

100%|██████████| 112165/112165 [00:03<00:00, 36842.36it/s]

HealthCareMagic (Chat Format) Elderly Samples: 0


### Re-unifying, Final Cleaning, and Exporting Datasets with New Entries

In [ ]:
print("\n--> Re-combining All Datasets with new entries...")
final_df = pd.concat(processed_datasets, ignore_index=True)

# Drop missing values and duplicates across input/response
final_df.dropna(subset=["input", "response"], inplace=True)
final_df.drop_duplicates(subset=["input", "response"], inplace=True)

# Remove trivial/extremely short inputs or outputs
final_df = final_df[(final_df['input'].str.len() > 10) & (final_df['response'].str.len() > 10)]

print(f"\nUpdated Final Consolidated Elderly Dataset Size: {len(final_df)} records")
print("\nUpdated Sample Distribution by Source:")
print(final_df['source'].value_counts())

# Re-export to JSON and Hugging Face Dataset format
final_df.to_json("rafael.json", orient="records", indent=2)
hf_dataset = Dataset.from_pandas(final_df)
hf_dataset.save_to_disk("./elderly_dataset_hf")

print("\nRe-saved output files with all new entries:")
print("1. rafael.json")
print("2. ./elderly_dataset_hf (Hugging Face Dataset folder)")


--> Re-combining All Datasets with new entries...

Updated Final Consolidated Elderly Dataset Size: 6093 records

Updated Sample Distribution by Source:
source
HealthCareMagic                       4971
MedQuAD                               1112
Agent Generated Medical Conditions      10
Name: count, dtype: int64


Saving the dataset (0/1 shards):   0%|          | 0/6093 [00:00<?, ? examples/s]


Re-saved output files with all new entries:
1. rafael.json
2. ./elderly_dataset_hf (Hugging Face Dataset folder)


### Visualization: Distribution of Medical Conditions by Source

### Consolidating and Cleaning Datasets (using `clean_text`)

In [ ]:
!pip install -q datasets pandas

import json
import os
from datasets import load_dataset

# Local workspace folder in Colab runtime
OUTPUT_DIR = "/content/medical_datasets"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output files will be saved to local session folder: {OUTPUT_DIR}\n")

Output files will be saved to local session folder: /content/medical_datasets



In [ ]:
def save_json(data, filename):
    filepath = os.path.join(OUTPUT_DIR, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved {len(data):,} records to '{filepath}'")


combined_medical_data = []
dataset_counts = {}

1. PubMedQA

In [33]:
print("Processing 1/4: PubMedQA...")
try:
    pubmed_ds = load_dataset("fedml/PubMedQA_instruction", split="train")
    pubmedqa_list = [
        {
            "source": "PubMedQA",
            "instruction": clean_text(row.get("instruction", "")), # Using clean_text
            "input": clean_text(row.get("input", "")), # Using clean_text
            "response": clean_text(row.get("output", "")), # Renamed output to response, using clean_text
            "category": "Medical Q&A" # Added category
        }
        for row in pubmed_ds
    ]
    save_json(pubmedqa_list, "PubMedQA.json")
    combined_medical_data.extend(pubmedqa_list)
    dataset_counts["PubMedQA"] = len(pubmedqa_list)
except Exception as e:
    print(f"Failed to process PubMedQA: {e}")

Processing 1/4: PubMedQA...
Saved 272,518 records to '/content/medical_datasets/PubMedQA.json'


# MedDialog (Direct working HF Repository: khoaliamle/MedDialog-EN-100k)

In [38]:
print("\nProcessing 3/4: MedDialog...")
try:
    meddialog_ds = load_dataset("khoaliamle/MedDialog-EN-100k", split="train")
    meddialog_list = [
        {
            "instruction": "Respond helpfully and accurately to the medical concern presented.",
            "input": str(
                row.get("input") or row.get("description") or ""
            ).strip(),
            "output": str(
                row.get("output") or row.get("formal_response") or ""
            ).strip(),
        }
        for row in meddialog_ds
    ]
    save_json(meddialog_list, "MedDialog.json")
    combined_medical_data.extend(meddialog_list)
    dataset_counts["MedDialog"] = len(meddialog_list)
except Exception as e:
    print(f"Failed to process MedDialog: {e}")


Processing 3/4: MedDialog...
Saved 112,165 records to '/content/medical_datasets/MedDialog.json'


# 4. HealthCareMagic QA

In [ ]:
print("\nProcessing 4/4: HealthCareMagic QA...")
try:
    hcm_ds = load_dataset(
        "lavita/ChatDoctor-HealthCareMagic-100k", split="train"
    )
    healthcaremagic_list = [
        {
            "source": "HealthCareMagicQA",
            "instruction": clean_text(row.get(
                "instruction",
                "If you are a doctor, please answer the medical questions according to the query below.",
            )),
            "input": clean_text(row.get("input", "")),
            "response": clean_text(row.get("output", "")), # Renamed output to response, using clean_text
            "category": "Medical Dialogue" # Added category
        }
        for row in hcm_ds
    ]
    save_json(healthcaremagic_list, "HealthCareMagicQA.json")
    combined_medical_data.extend(healthcaremagic_list)
    dataset_counts["HealthCareMagicQA"] = len(healthcaremagic_list)
except Exception as e:
    print(f"Failed to process HealthCareMagic QA: {e}")


Processing 4/4: HealthCareMagic QA...
Saved 112,165 records to '/content/medical_datasets/HealthCareMagicQA.json'


In [42]:
print("\nProcessing 2/4: MedQuAD...")
try:
    medquad_ds = load_dataset(
        "keivalya/MedQuad-MedicalQnADataset", split="train"
    )
    medquad_list = [
        {
            "instruction": "Answer the following medical question accurately.",
            "input": row.get("Question", "").strip(),
            "output": row.get("Answer", "").strip(),
        }
        for row in medquad_ds
    ]
    save_json(medquad_list, "MedQuAD.json")
    combined_medical_data.extend(medquad_list)
    dataset_counts["MedQuAD"] = len(medquad_list)
except Exception as e:
    print(f"Failed to process MedQuAD: {e}")


Processing 2/4: MedQuAD...
Saved 16,407 records to '/content/medical_datasets/MedQuAD.json'


In [45]:
import re

# Bad answer prefixes to filter out generic or uninformative metadata templates
BAD_ANSWER_PREFIXES = (
    "These resources address",
    "These resources from MedlinePlus",
    "For more information, go to",
    "For more information go to",
    "Read more on MedlinePlus",
)


def clean_text(text):
    """Normalizes whitespace and strips unwanted leading/trailing artifacts."""
    if not text:
        return ""
    # Replace newlines, tabs, and multiple spaces with a single space
    text = re.sub(r"\s+", " ", str(text))
    # Remove leading non-alphanumeric noise if present
    text = text.strip()
    return text


print("\nProcessing 2/4: MedQuAD (Cleaned & Preprocessed)...")
try:
    medquad_ds = load_dataset(
        "keivalya/MedQuad-MedicalQnADataset", split="train"
    )
    medquad_list = []

    for row in medquad_ds:
        question = clean_text(row.get("Question", ""))
        answer = clean_text(row.get("Answer", ""))

        # Quality check: Skip empty fields or extremely short junk text
        if len(question) < 5 or len(answer) < 10:
            continue

        # Filter out generic or non-informative default answer boilerplate
        if answer.startswith(BAD_ANSWER_PREFIXES):
            continue

        medquad_list.append({
            "instruction": "Answer the following medical question accurately.",
            "input": question,
            "output": answer,
        })

    save_json(medquad_list, "MedQuAD.json")
    combined_medical_data.extend(medquad_list)
    dataset_counts["MedQuAD"] = len(medquad_list)

except Exception as e:
    print(f"Failed to process MedQuAD: {e}")


Processing 2/4: MedQuAD (Cleaned & Preprocessed)...
Saved 15,320 records to '/content/medical_datasets/MedQuAD.json'


# 5. Merge all into medical.json

In [46]:
print("\nMerging all datasets into medical.json...")
save_json(combined_medical_data, "medical.json")

# Summary Table
print("\n" + "=" * 45)
print("PROCESSING SUMMARY")
print("=" * 45)
for name, count in dataset_counts.items():
    print(f"{name:<22}: {count:>8,} samples")
print("-" * 45)
print(
    f"{'Total (medical.json)':<22}: {len(combined_medical_data):>8,} samples"
)
print("=" * 45)


Merging all datasets into medical.json...
Saved 1,555,084 records to '/content/medical_datasets/medical.json'

PROCESSING SUMMARY
PubMedQA              :  272,518 samples
HealthCareMagicQA     :  112,165 samples
MedDialog             :  112,165 samples
MedQuAD               :   15,320 samples
---------------------------------------------
Total (medical.json)  : 1,555,084 samples
